In [2]:
import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
import json
import time
import xarray as xr
import scipy.signal as signal
import glob
import os
import xgboost as xgb
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score, StratifiedKFold, permutation_test_score, GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score,make_scorer,balanced_accuracy_score, f1_score

import matplotlib.gridspec as gridspec
from matplotlib.ticker import FormatStrFormatter
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import seaborn as sns

from matplotlib.ticker import MultipleLocator


data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
reference_dir = data_dir + '/CHCH_v6_reference/'
reference_mask_dir = reference_dir + 'processed{}/'
model_save_dir = reference_dir + 'models/'

model_mask_dir = data_dir + '/CHCH_v6_model/{}/model_processed_{}/'

output_dir = data_dir + '/CHCH_v6_model/figures/'


In [3]:
def printStats(slwc_t_ref, slwc_t_mod, pks_t, target_name):
    
    ## Computing stats
    print('{:.0f} cloudy time steps (with peaks detected), out of {} time steps ({:.1f}%)'.format(pks_t.sum(),len(pks_t),100*pks_t.mean()))
    print('Ref: {:.0f} time steps have {}, out of {} total time steps ({:.1f}%)'.format(slwc_t_ref.sum(),target_name,len(slwc_t_ref),100*slwc_t_ref.mean()))
    print('Ref: {:.0f} time steps have {}, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_ref[pks_t].sum(),target_name,len(slwc_t_ref[pks_t]),100*slwc_t_ref[pks_t].mean()))
    print('Model: {:.0f} time steps have {}, out of {} total time steps ({:.1f}%)'.format(slwc_t_mod.sum(),target_name,len(slwc_t_mod),100*slwc_t_mod.mean()))
    print('Model: {:.0f} time steps have {}, out of {} cloudy time steps ({:.1f}%)'.format(slwc_t_mod[pks_t].sum(),target_name,len(slwc_t_mod[pks_t]),100*slwc_t_mod[pks_t].mean()))

    print('Precision/Recall of {} classification, examining all time steps: {:.3f}/{:.3f}'.format(target_name,precision_score(slwc_t_ref, slwc_t_mod),recall_score(slwc_t_ref, slwc_t_mod)))
    print('Precision/Recall of {} classification, examining cloudy time steps: {:.3f}/{:.3f}'.format(target_name,precision_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t]),recall_score(slwc_t_ref[pks_t], slwc_t_mod[pks_t])))


def getStats(date, model_name):

    ## Opening files
    infile_ref = reference_mask_dir.format(date[:-2])+str(date)+'.nc'
    # print('  <--- ' + infile_ref)
    ds_ref = xr.open_dataset(infile_ref)

    infile_mod = model_mask_dir.format(model_name,date[:-2])+str(date)+'.nc'
    # print('  <--- ' + infile_mod)
    ds_mod = xr.open_dataset(infile_mod)

    ## Variables for comparison are different for each model
    if model_name in ['model_003']:  # G22-Christchurch output compared to reference mask SLWC
        ref_var = ds_ref.layer_slwc
        mod_var_t = ds_mod.slwc_t
    elif model_name in ['AG_XGBoost']: # G22-Davis output compared to reference mask SLCC
        ref_var = ds_ref.layer_slcc
        mod_var_t = ds_mod.slwc_t # (actually still SLCC)

    ref_var_t = ref_var.max(dim='level')

    class_pks = ds_mod.classified_peaks.astype(float)
    class_pks = class_pks.where(class_pks>=0,np.nan )
    pks_t = ~class_pks.max(dim='level').isnull()  # True if peaks are present, false if no peaks present

    ds_ref.close()
    ds_mod.close()

    return ref_var_t, mod_var_t, pks_t

In [14]:
## Accuracy scores for case study 1

m3_var_ref_t, m3_var_mod_t, m3_pks_t = getStats('20210518', 'model_003')
print('G22-Christchurch:')
printStats(m3_var_ref_t, m3_var_mod_t, m3_pks_t, 'SLWC')

mAG_var_ref_t, mAG_var_mod_t, mAG_pks_t = getStats('20210518', 'AG_XGBoost')
print('G22-Davis:')
printStats(mAG_var_ref_t, mAG_var_mod_t, mAG_pks_t, 'SLCC')

G22-Christchurch:
188 cloudy time steps (with peaks detected), out of 288 time steps (65.3%)
Ref: 155 time steps have SLWC, out of 288 total time steps (53.8%)
Ref: 152 time steps have SLWC, out of 188 cloudy time steps (80.9%)
Model: 157 time steps have SLWC, out of 288 total time steps (54.5%)
Model: 157 time steps have SLWC, out of 188 cloudy time steps (83.5%)
Precision/Recall of SLWC classification, examining all time steps: 0.943/0.955
Precision/Recall of SLWC classification, examining cloudy time steps: 0.943/0.974
G22-Davis:
188 cloudy time steps (with peaks detected), out of 288 time steps (65.3%)
Ref: 172 time steps have SLCC, out of 288 total time steps (59.7%)
Ref: 159 time steps have SLCC, out of 188 cloudy time steps (84.6%)
Model: 34 time steps have SLCC, out of 288 total time steps (11.8%)
Model: 34 time steps have SLCC, out of 188 cloudy time steps (18.1%)
Precision/Recall of SLCC classification, examining all time steps: 0.941/0.186
Precision/Recall of SLCC classifica

In [13]:
## Accuracy scores for case study 2

m3_var_ref_t, m3_var_mod_t, m3_pks_t = getStats('20210605', 'model_003')
print('G22-Christchurch:')
printStats(m3_var_ref_t, m3_var_mod_t, m3_pks_t, 'SLWC')

mAG_var_ref_t, mAG_var_mod_t, mAG_pks_t = getStats('20210605', 'AG_XGBoost')
print('G22-Davis:')
printStats(mAG_var_ref_t, mAG_var_mod_t, mAG_pks_t, 'SLCC')

G22-Christchurch:
283 cloudy time steps (with peaks detected), out of 288 time steps (98.3%)
Ref: 60 time steps have SLWC, out of 288 total time steps (20.8%)
Ref: 56 time steps have SLWC, out of 283 cloudy time steps (19.8%)
Model: 70 time steps have SLWC, out of 288 total time steps (24.3%)
Model: 70 time steps have SLWC, out of 283 cloudy time steps (24.7%)
Precision/Recall of SLWC classification, examining all time steps: 0.643/0.750
Precision/Recall of SLWC classification, examining cloudy time steps: 0.643/0.804
G22-Davis:
283 cloudy time steps (with peaks detected), out of 288 time steps (98.3%)
Ref: 68 time steps have SLCC, out of 288 total time steps (23.6%)
Ref: 64 time steps have SLCC, out of 283 cloudy time steps (22.6%)
Model: 50 time steps have SLCC, out of 288 total time steps (17.4%)
Model: 50 time steps have SLCC, out of 283 cloudy time steps (17.7%)
Precision/Recall of SLCC classification, examining all time steps: 0.120/0.088
Precision/Recall of SLCC classification, 

In [15]:
### Concatenates every day, then gets total accuracy.

dates = [file[-11:-3].replace('-','') for file in sorted(glob.glob(model_mask_dir.format('model_003', '*')+'/*.nc'))]

model_dict = {'model_003':'SLWC', #  G22-Christchurch
            'AG_XGBoost':'SLCC'}  #  G22-Davis

for (model_name,target_name) in model_dict.items():
    var_ref_t_all = []
    var_mod_t_all = []
    pks_t_all = []

    for date in dates:
        # print(date)
        var_ref_t, var_mod_t, pks_t = getStats(date, model_name)
        var_ref_t_all.append(var_ref_t)
        var_mod_t_all.append(var_mod_t)
        pks_t_all.append(pks_t)

    var_ref_t_ds = xr.concat(var_ref_t_all, dim='time')
    var_mod_t_ds = xr.concat(var_mod_t_all, dim='time')
    pks_t_ds = xr.concat(pks_t_all, dim='time')

    print(model_name)
    print(' ---> Full dataset')
    printStats(var_ref_t_ds, var_mod_t_ds, pks_t_ds, target_name)




model_003
 ---> Full dataset
31708 cloudy time steps (with peaks detected), out of 66240 time steps (47.9%)
Ref: 10112 time steps have SLWC, out of 66240 total time steps (15.3%)
Ref: 8729 time steps have SLWC, out of 31708 cloudy time steps (27.5%)
Model: 8570 time steps have SLWC, out of 66240 total time steps (12.9%)
Model: 8570 time steps have SLWC, out of 31708 cloudy time steps (27.0%)
Precision/Recall of SLWC classification, examining all time steps: 0.891/0.756
Precision/Recall of SLWC classification, examining cloudy time steps: 0.891/0.875
AG_XGBoost
 ---> Full dataset
31708 cloudy time steps (with peaks detected), out of 66240 time steps (47.9%)
Ref: 11299 time steps have SLCC, out of 66240 total time steps (17.1%)
Ref: 9464 time steps have SLCC, out of 31708 cloudy time steps (29.8%)
Model: 11179 time steps have SLCC, out of 66240 total time steps (16.9%)
Model: 11179 time steps have SLCC, out of 31708 cloudy time steps (35.3%)
Precision/Recall of SLCC classification, exami